In [ ]:
import duckdb

In [ ]:
con = duckdb.connect("data/silver_3nf.duckdb")

con.execute('''
-- ============================================================
-- SEQUENCES for surrogate keys
-- ============================================================
CREATE SEQUENCE seq_dim_team        START 1;
CREATE SEQUENCE seq_dim_stadium     START 1;
CREATE SEQUENCE seq_dim_competition START 1;
CREATE SEQUENCE seq_fact_team_match START 1;

-- ============================================================
-- dim_date
-- Date key as YYYYMMDD integer is the Kimball convention -
-- human-readable, sorts correctly, joins cheaply.
-- ============================================================
CREATE TABLE dim_date (
    date_key            INTEGER     PRIMARY KEY,        -- e.g. 20231028
    full_date           DATE        NOT NULL,
    day_of_week         VARCHAR     NOT NULL,           -- 'Saturday'
    day_of_month        SMALLINT    NOT NULL,
    month_number        SMALLINT    NOT NULL,
    month_name          VARCHAR     NOT NULL,
    quarter             SMALLINT    NOT NULL,
    year                SMALLINT    NOT NULL,
    is_weekend          BOOLEAN     NOT NULL,
    -- Rugby-specific business flags
    is_modern_era       BOOLEAN     NOT NULL,           -- year >= 1995
    is_world_cup_year   BOOLEAN     NOT NULL
);

-- ============================================================
-- dim_team  (SCD Type 2)
-- Same physical table joined twice from the fact: once as team,
-- once as opponent. Surrogate key changes per version; natural
-- key (team_code) stays constant across versions of the same team.
-- ============================================================
CREATE TABLE dim_team (
    team_sk             BIGINT      PRIMARY KEY DEFAULT nextval('seq_dim_team'),
    team_code           VARCHAR     NOT NULL,           -- natural key, e.g. 'RSA', 'NZL'
    team_name           VARCHAR     NOT NULL,           -- 'South Africa'
    nickname            VARCHAR,                        -- 'Springboks'
    country             VARCHAR     NOT NULL,
    --union_name          VARCHAR,                        -- 'SA Rugby Union'
    --coach               VARCHAR,                        -- Type 2 tracked attribute
    --world_ranking       SMALLINT,                       -- Type 2 tracked (snapshot at match time)
    -- SCD2 housekeeping
    effective_from      DATE        NOT NULL,
    effective_to        DATE        NOT NULL DEFAULT DATE '9999-12-31',
    is_current          BOOLEAN     NOT NULL DEFAULT TRUE,
    UNIQUE (team_code, effective_from)
);

-- ============================================================
-- dim_stadium  (Type 1 - stadiums rarely change meaningfully)
-- ============================================================
CREATE TABLE dim_stadium (
    stadium_sk          BIGINT      PRIMARY KEY DEFAULT nextval('seq_dim_stadium'),
    stadium_code        VARCHAR     NOT NULL UNIQUE,    -- natural key
    stadium_name        VARCHAR     NOT NULL,
    city                VARCHAR     NOT NULL,
    country             VARCHAR     NOT NULL,
    --capacity            INTEGER,
    --altitude_m          INTEGER,                        -- rugby-relevant
    --avg_temp_c          DECIMAL(4,1)                   -- typical/climate
);

-- ============================================================
-- dim_competition  (Type 1)
-- ============================================================
CREATE TABLE dim_competition (
    competition_sk      BIGINT      PRIMARY KEY DEFAULT nextval('seq_dim_competition'),
    competition_code    VARCHAR     NOT NULL,           -- e.g. 'RWC2023'
    competition_name    VARCHAR     NOT NULL,           -- 'Rugby World Cup 2023'
    competition_type    VARCHAR     NOT NULL,           -- 'World Cup', 'Rugby Championship', 'Test Series', 'Friendly'
    stage               VARCHAR,                        -- 'Pool', 'QF', 'SF', 'Final', 'Round Robin'
    is_knockout         BOOLEAN     NOT NULL,
    season_year         SMALLINT    NOT NULL,
    UNIQUE (competition_code, stage)
);

-- ============================================================
-- fact_team_match
-- Grain: ONE ROW PER TEAM PER MATCH
-- (two rows per real-world match, mirrored perspectives)
-- ============================================================
CREATE TABLE fact_team_match (
    -- Surrogate PK
    team_match_sk           BIGINT      PRIMARY KEY DEFAULT nextval('seq_fact_team_match'),

    -- Degenerate dimension: groups the two rows that represent the same match
    match_id                VARCHAR     NOT NULL,       -- e.g. 'RWC2023-FINAL' or a hash

    -- Foreign keys to dimensions
    date_key                INTEGER     NOT NULL REFERENCES dim_date(date_key),
    team_sk                 BIGINT      NOT NULL REFERENCES dim_team(team_sk),
    opponent_sk             BIGINT      NOT NULL REFERENCES dim_team(team_sk),
    stadium_sk              BIGINT      NOT NULL REFERENCES dim_stadium(stadium_sk),
    competition_sk          BIGINT      NOT NULL REFERENCES dim_competition(competition_sk),

    -- Fact-row context (not dimensions, but describe this row)
    venue_type              VARCHAR     NOT NULL,       -- 'home' | 'away' | 'neutral'
    --match_duration_minutes  SMALLINT,                   -- 80 unless extra time
    --attendance              INTEGER,                    -- match-level, redundant across the two rows but harmless

    -- Scoring (fully additive)
    points_for              SMALLINT    NOT NULL,
    points_against          SMALLINT    NOT NULL,
    --tries_for               SMALLINT    NOT NULL,
    --tries_against           SMALLINT    NOT NULL,
    --conversions_made        SMALLINT    NOT NULL,
    --conversions_attempted   SMALLINT    NOT NULL,
    --penalties_made          SMALLINT    NOT NULL,
    --penalties_attempted     SMALLINT    NOT NULL,
    --drop_goals_made         SMALLINT    NOT NULL,
    --drop_goals_attempted    SMALLINT    NOT NULL,

    -- Attack (team's own actions, additive)
    --metres_made             INTEGER,
    --runs                    SMALLINT,
    --passes                  SMALLINT,
    --offloads                SMALLINT,
    --carries_over_gainline   SMALLINT,
    --kicks_from_hand         SMALLINT,

    -- Defence
    tackles_made            SMALLINT,
    tackles_missed          SMALLINT,
    turnovers_won           SMALLINT,
    rucks_won               SMALLINT,
    mauls_won               SMALLINT,

    -- Set piece (numerators and denominators, NOT percentages)
    --scrums_won_on_own_feed  SMALLINT,
    --scrums_lost_on_own_feed SMALLINT,
    --lineouts_won            SMALLINT,
    --lineouts_lost           SMALLINT,

    -- Possession / territory
    -- Stored as percentages because that's how the source publishes them.
    -- Flag: NOT additive across rows - use AVG, not SUM, at query time.
    --possession_pct          DECIMAL(5,2),
    --territory_pct           DECIMAL(5,2),

    -- Discipline
    --penalties_conceded      SMALLINT,
    --yellow_cards            SMALLINT    NOT NULL DEFAULT 0,
    --red_cards               SMALLINT    NOT NULL DEFAULT 0,

    -- ETL metadata (Kimball "audit dimension lite")
    source_system           VARCHAR,
    loaded_at               TIMESTAMP   NOT NULL DEFAULT current_timestamp,

    -- Prevent the same team-perspective row being loaded twice
    UNIQUE (match_id, team_sk)
);

-- ============================================================
-- Role-playing views over dim_team
-- These let queries read naturally without aliasing every time.
-- ============================================================
CREATE VIEW dim_team_role_team AS
SELECT
    team_sk          AS team_sk,
    team_code        AS team_code,
    team_name        AS team_name,
    nickname         AS team_nickname,
    country          AS team_country,
    --coach            AS team_coach,
    --world_ranking    AS team_world_ranking,
    --is_tier_one      AS team_is_tier_one,
    is_current       AS team_is_current_version
FROM dim_team;

CREATE VIEW dim_team_role_opponent AS
SELECT
    team_sk          AS opponent_sk,
    team_code        AS opponent_code,
    team_name        AS opponent_name,
    nickname         AS opponent_nickname,
    country          AS opponent_country,
    coach            AS opponent_coach,
    world_ranking    AS opponent_world_ranking,
    is_tier_one      AS opponent_is_tier_one
FROM dim_team; ''')